# IPL Match Outcome Prediction — Support Vector Machine (SVM)
**Member 3 | Branch: member3-svm**

## Objective
Predict IPL match outcomes during the second innings (chase) 
using over-by-over game state features built from raw delivery data.

In [1]:
# ── IMPORTS ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve
)

print("All libraries imported successfully.")

All libraries imported successfully.


In [2]:
# ── LOAD RAW DATA ──────────────────────────────────────────────────
matches    = pd.read_csv('../data/matches.csv')
deliveries = pd.read_csv('../data/deliveries.csv')

print(f"matches.csv    : {matches.shape[0]} rows, {matches.shape[1]} columns")
print(f"deliveries.csv : {deliveries.shape[0]} rows, {deliveries.shape[1]} columns")

matches.csv    : 636 rows, 18 columns
deliveries.csv : 150460 rows, 21 columns


In [3]:
# ── EXPLORE MATCHES ────────────────────────────────────────────────
print("=== MATCHES - First 3 rows ===")
display(matches.head(3))

print("\n=== MATCHES - Shape ===")
print(matches.shape)

print("\n=== MATCHES - Column Names ===")
print(list(matches.columns))

print("\n=== MATCHES - Missing Values ===")
print(matches.isnull().sum())

=== MATCHES - First 3 rows ===


,id,season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
0,1,2017,Hyderabad,2017-04-05,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
1,2,2017,Pune,2017-04-06,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,field,normal,0,Rising Pune Supergiant,0,7,SPD Smith,Maharashtra Cricket Association Stadium,A Nand Kishore,S Ravi,NaN
2,3,2017,Rajkot,2017-04-07,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,normal,0,Kolkata Knight Riders,0,10,CA Lynn,Saurashtra Cricket Association Stadium,Nitin Menon,CK Nandan,NaN



=== MATCHES - Shape ===
(636, 18)

=== MATCHES - Column Names ===
['id', 'season', 'city', 'date', 'team1', 'team2', 'toss_winner', 'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs', 'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2', 'umpire3']

=== MATCHES - Missing Values ===
id                   0
season               0
city                 7
date                 0
team1                0
team2                0
toss_winner          0
toss_decision        0
result               0
dl_applied           0
winner               3
win_by_runs          0
win_by_wickets       0
player_of_match      3
venue                0
umpire1              1
umpire2              1
umpire3            636
dtype: int64


In [4]:
# ── EXPLORE DELIVERIES ─────────────────────────────────────────────
print("=== DELIVERIES - First 3 rows ===")
display(deliveries.head(3))

print("\n=== DELIVERIES - Shape ===")
print(deliveries.shape)

print("\n=== DELIVERIES - Column Names ===")
print(list(deliveries.columns))

print("\n=== DELIVERIES - Missing Values ===")
print(deliveries.isnull().sum())

=== DELIVERIES - First 3 rows ===


,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,4,0,4,NaN,NaN,NaN



=== DELIVERIES - Shape ===
(150460, 21)

=== DELIVERIES - Column Names ===
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batsman', 'non_striker', 'bowler', 'is_super_over', 'wide_runs', 'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs', 'batsman_runs', 'extra_runs', 'total_runs', 'player_dismissed', 'dismissal_kind', 'fielder']

=== DELIVERIES - Missing Values ===
match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batsman                  0
non_striker              0
bowler                   0
is_super_over            0
wide_runs                0
bye_runs                 0
legbye_runs              0
noball_runs              0
penalty_runs             0
batsman_runs             0
extra_runs               0
total_runs               0
player_dismissed    143022
dismissal_kind      143022
fielder             145091
dtype: int64


In [5]:
# ── CLEAN MATCHES ──────────────────────────────────────────────────
# Based on exploration:
# - Drop umpire columns (not relevant to prediction)
# - Drop player_of_match (not a predictive feature)
# - Drop rows where winner is NaN (no-result matches)
# - Fill missing city with 'Unknown'
# - Create target column: 1 if team1 won, 0 if team2 won

matches.drop(columns=['umpire1', 'umpire2', 'umpire3',
                       'player_of_match'], inplace=True)

matches.dropna(subset=['winner'], inplace=True)

matches['city'] = matches['city'].fillna('Unknown')

matches['team1_won'] = (matches['winner'] == matches['team1']).astype(int)

print(f"Matches after cleaning : {matches.shape[0]} rows")
print(f"Columns remaining      : {matches.shape[1]}")
print(f"\nTarget distribution:")
print(matches['team1_won'].value_counts())

Matches after cleaning : 633 rows
Columns remaining      : 15

Target distribution:
team1_won
0    349
1    284
Name: count, dtype: int64


In [6]:
# ── CLEAN DELIVERIES ───────────────────────────────────────────────
# Based on exploration:
# - Keep inning 2 only (chase innings — this is what we predict)
# - Remove super overs (special case, not regular play)
# - player_dismissed NaN means no wicket — mark as 0/1 wicket column

deliveries_clean = deliveries[
    (deliveries['inning'] == 2) &
    (deliveries['is_super_over'] == 0)
].copy()

# Create binary wicket column
# NaN in player_dismissed = no wicket = 0
# Any name in player_dismissed = wicket fell = 1
deliveries_clean['wicket'] = deliveries_clean['player_dismissed'].notna().astype(int)

print(f"Deliveries after cleaning : {deliveries_clean.shape[0]} rows")
print(f"Unique matches            : {deliveries_clean['match_id'].nunique()}")
print(f"\nTotal wickets in dataset  : {deliveries_clean['wicket'].sum()}")

Deliveries after cleaning : 72350 rows
Unique matches            : 634

Total wickets in dataset  : 3555


In [7]:
# ── BUILD OVER-BY-OVER FEATURES FROM INNING 2 ──────────────────────
# Each row = game state at end of one over in one match
# This tells us: at this point in the chase, what is happening?

over_stats = deliveries_clean.groupby(['match_id', 'over']).agg(
    runs_in_over    = ('total_runs',   'sum'),
    wickets_in_over = ('wicket',       'sum'),
    extras_in_over  = ('extra_runs',   'sum'),
    fours_in_over   = ('batsman_runs', lambda x: (x == 4).sum()),
    sixes_in_over   = ('batsman_runs', lambda x: (x == 6).sum()),
    balls_in_over   = ('ball',         'count'),
).reset_index()

# Sort by match and over
over_stats = over_stats.sort_values(['match_id', 'over']).reset_index(drop=True)

print(f"Over-by-over stats shape : {over_stats.shape}")
print(f"\nSample:")
display(over_stats.head(10))

Over-by-over stats shape : (11795, 8)

Sample:


,match_id,over,runs_in_over,wickets_in_over,extras_in_over,fours_in_over,sixes_in_over,balls_in_over
0,1,1,11,0,0,2,0,6
1,1,2,1,0,0,0,0,6
2,1,3,16,0,0,2,1,6
3,1,4,15,0,1,1,1,7
4,1,5,5,0,1,1,0,7
5,1,6,6,1,0,1,0,6
6,1,7,7,1,0,0,1,6
7,1,8,13,0,0,2,0,6
8,1,9,11,0,1,0,1,7
9,1,10,13,0,1,2,0,7


In [8]:
# ── ADD CUMULATIVE STATS ───────────────────────────────────────────
# At each over we need to know TOTAL so far, not just that over
# Example: by over 10, team scored 98 runs total, lost 3 wickets total

over_stats['cum_runs']     = over_stats.groupby('match_id')['runs_in_over'].cumsum()
over_stats['cum_wickets']  = over_stats.groupby('match_id')['wickets_in_over'].cumsum()
over_stats['cum_extras']   = over_stats.groupby('match_id')['extras_in_over'].cumsum()
over_stats['cum_fours']    = over_stats.groupby('match_id')['fours_in_over'].cumsum()
over_stats['cum_sixes']    = over_stats.groupby('match_id')['sixes_in_over'].cumsum()

print("Cumulative stats added!")
print(f"Shape: {over_stats.shape}")
display(over_stats.head(10))

Cumulative stats added!
Shape: (11795, 13)


,match_id,over,runs_in_over,wickets_in_over,extras_in_over,fours_in_over,sixes_in_over,balls_in_over,cum_runs,cum_wickets,cum_extras,cum_fours,cum_sixes
0,1,1,11,0,0,2,0,6,11,0,0,2,0
1,1,2,1,0,0,0,0,6,12,0,0,2,0
2,1,3,16,0,0,2,1,6,28,0,0,4,1
3,1,4,15,0,1,1,1,7,43,0,1,5,2
4,1,5,5,0,1,1,0,7,48,0,2,6,2
5,1,6,6,1,0,1,0,6,54,1,2,7,2
6,1,7,7,1,0,0,1,6,61,2,2,7,3
7,1,8,13,0,0,2,0,6,74,2,2,9,3
8,1,9,11,0,1,0,1,7,85,2,3,9,4
9,1,10,13,0,1,2,0,7,98,2,4,11,4


In [9]:
# ── ADD TARGET SCORE FROM INNING 1 ────────────────────────────────
# Chasing team needs to know what score they are chasing
# Target = inning 1 total runs + 1

inn1_totals = deliveries[
    (deliveries['inning'] == 1) &
    (deliveries['is_super_over'] == 0)
].groupby('match_id')['total_runs'].sum().reset_index()

inn1_totals.columns = ['match_id', 'target_runs']
inn1_totals['target_runs'] = inn1_totals['target_runs'] + 1

# Merge target into over stats
over_stats = over_stats.merge(inn1_totals, on='match_id', how='left')

print("Target score added!")
display(over_stats.head(5))

Target score added!


,match_id,over,runs_in_over,wickets_in_over,extras_in_over,fours_in_over,sixes_in_over,balls_in_over,cum_runs,cum_wickets,cum_extras,cum_fours,cum_sixes,target_runs
0,1,1,11,0,0,2,0,6,11,0,0,2,0,208
1,1,2,1,0,0,0,0,6,12,0,0,2,0,208
2,1,3,16,0,0,2,1,6,28,0,0,4,1,208
3,1,4,15,0,1,1,1,7,43,0,1,5,2,208
4,1,5,5,0,1,1,0,7,48,0,2,6,2,208


In [10]:
# ── CALCULATE MATCH SITUATION FEATURES ────────────────────────────
# These features tell us HOW the chase is going at each over
# This is what makes prediction meaningful and powerful

over_stats['runs_needed']  = over_stats['target_runs'] - over_stats['cum_runs']
over_stats['balls_left']   = (20 - over_stats['over']) * 6
over_stats['wickets_left'] = 10 - over_stats['cum_wickets']
over_stats['current_rr']   = over_stats['cum_runs'] / over_stats['over']
over_stats['required_rr']  = (over_stats['runs_needed'] / 
                               over_stats['balls_left'].replace(0, 1)) * 6

print("Match situation features added!")
print(f"Shape: {over_stats.shape}")
display(over_stats.head(5))

Match situation features added!
Shape: (11795, 19)


,match_id,over,runs_in_over,wickets_in_over,extras_in_over,fours_in_over,sixes_in_over,balls_in_over,cum_runs,cum_wickets,cum_extras,cum_fours,cum_sixes,target_runs,runs_needed,balls_left,wickets_left,current_rr,required_rr
0,1,1,11,0,0,2,0,6,11,0,0,2,0,208,197,114,10,11.000000,10.368421
1,1,2,1,0,0,0,0,6,12,0,0,2,0,208,196,108,10,6.000000,10.888889
2,1,3,16,0,0,2,1,6,28,0,0,4,1,208,180,102,10,9.333333,10.588235
3,1,4,15,0,1,1,1,7,43,0,1,5,2,208,165,96,10,10.750000,10.312500
4,1,5,5,0,1,1,0,7,48,0,2,6,2,208,160,90,10,9.600000,10.666667


In [11]:
# ── MERGE WITH MATCH RESULT (TARGET LABEL) ────────────────────────
# We need to know who actually won each match
# This becomes our target: 1 = chasing team won, 0 = chasing team lost

# Get the chasing team (batting team in inning 2) per match
chasing_team = deliveries_clean.groupby('match_id')['batting_team'].first().reset_index()
chasing_team.columns = ['match_id', 'chasing_team']

# Merge chasing team into over stats
over_stats = over_stats.merge(chasing_team, on='match_id', how='left')

# Merge match result
over_stats = over_stats.merge(
    matches[['id', 'winner', 'team1_won', 'toss_winner', 'toss_decision', 'city', 'venue', 'season']],
    left_on='match_id', right_on='id', how='left'
)

# Target: did the chasing team win?
over_stats['chasing_team_won'] = (over_stats['winner'] == over_stats['chasing_team']).astype(int)

print(f"Shape after merge : {over_stats.shape}")
print(f"\nTarget distribution:")
print(over_stats['chasing_team_won'].value_counts())
print(f"\nChase win rate: {over_stats['chasing_team_won'].mean()*100:.1f}%")
display(over_stats.head(3))

Shape after merge : (11795, 29)

Target distribution:
chasing_team_won
1    6263
0    5532
Name: count, dtype: int64

Chase win rate: 53.1%


,match_id,over,runs_in_over,wickets_in_over,extras_in_over,fours_in_over,sixes_in_over,balls_in_over,cum_runs,cum_wickets,...,chasing_team,id,winner,team1_won,toss_winner,toss_decision,city,venue,season,chasing_team_won
0,1,1,11,0,0,2,0,6,11,0,...,Royal Challengers Bangalore,1.0,Sunrisers Hyderabad,1.0,Royal Challengers Bangalore,field,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",2017.0,0
1,1,2,1,0,0,0,0,6,12,0,...,Royal Challengers Bangalore,1.0,Sunrisers Hyderabad,1.0,Royal Challengers Bangalore,field,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",2017.0,0
2,1,3,16,0,0,2,1,6,28,0,...,Royal Challengers Bangalore,1.0,Sunrisers Hyderabad,1.0,Royal Challengers Bangalore,field,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",2017.0,0


In [12]:
print("\n=== MATCHES - Column Names ===")
print(list(over_stats.columns))


=== MATCHES - Column Names ===
['match_id', 'over', 'runs_in_over', 'wickets_in_over', 'extras_in_over', 'fours_in_over', 'sixes_in_over', 'balls_in_over', 'cum_runs', 'cum_wickets', 'cum_extras', 'cum_fours', 'cum_sixes', 'target_runs', 'runs_needed', 'balls_left', 'wickets_left', 'current_rr', 'required_rr', 'chasing_team', 'id', 'winner', 'team1_won', 'toss_winner', 'toss_decision', 'city', 'venue', 'season', 'chasing_team_won']
